# 02 - Data Preparation and LSTM Windowing


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
data_path = "../data/raw/continuous_dataset.csv"

df = pd.read_csv(
    data_path,
    parse_dates=['datetime'],
    index_col='datetime'
).sort_index()


In [ ]:
df['hour'] = df.index.hour
df['dayofweek'] = df.index.day_of_week
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

feature_cols = [
    'nat_demand',
    'T2M_toc', 'T2M_san', 'T2M_dav',
    'hour', 'dayofweek', 'is_weekend',
    'holiday', 'school'
]

target_col = 'nat_demand'

df_model = df[feature_cols].copy()
df_model.head()

In [ ]:
n = len(df_model)
train_end = int(n * 0.70)
validation_end = train_end + int(n * 0.15)

df_train = df_model.iloc[:train_end]
df_validation = df_model.iloc[train_end:validation_end]
df_test = df_model.iloc[validation_end:]

print("Train:", df_train.index.min(), "->", df_train.index.max(), "| rows:", len(df_train))
print("Validation:", df_validation.index.min(), "->", df_validation.index.max(), "| rows:", len(df_validation))
print("Test:", df_test.index.min(), "->", df_test.index.max(), "| rows:", len(df_test))


#### Scaling


In [ ]:
X_train = df_train[feature_cols].values
y_train = df_train[[target_col]].values   # double brackets keep the target 2D

X_validation = df_validation[feature_cols].values
y_validation = df_validation[[target_col]].values

X_test = df_test[feature_cols].values
y_test = df_test[[target_col]].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)   # fit only on train
y_train_scaled = scaler_y.fit_transform(y_train)

X_validation_scaled = scaler_X.transform(X_validation)
y_validation_scaled = scaler_y.transform(y_validation)

X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

print(X_train_scaled.shape, y_train_scaled.shape, X_validation_scaled.shape, y_validation_scaled.shape, X_test_scaled.shape, y_test_scaled.shape)


In [ ]:
def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])  # [t-seq_len, ..., t-1]
        ys.append(y[i])            # value at t
    return np.array(Xs), np.array(ys)


In [ ]:
sequence_length = 24 * 7  # 7 days of history (168 hours)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, sequence_length)
X_validation_seq, y_validation_seq = create_sequences(X_validation_scaled, y_validation_scaled, sequence_length)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, sequence_length)

print("X_train_seq:", X_train_seq.shape)
print("y_train_seq:", y_train_seq.shape)
print("X_validation_seq:", X_validation_seq.shape)
print("y_validation_seq:", y_validation_seq.shape)
print("X_test_seq:", X_test_seq.shape)
print("y_test_seq:", y_test_seq.shape)


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32).to(device)

X_validation_tensor = torch.tensor(X_validation_seq, dtype=torch.float32).to(device)
y_validation_tensor = torch.tensor(y_validation_seq, dtype=torch.float32).to(device)

X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_seq, dtype=torch.float32).to(device)

X_train_tensor.shape, y_train_tensor.shape


In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
validation_dataset = TensorDataset(X_validation_tensor, y_validation_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size, shuffle=False)


In [ ]:
for X_batch, y_batch in train_loader:
    print(X_batch.shape, y_batch.shape)
    break
print(len(train_loader))

In [ ]:
import torch.nn as nn 

class LSTMForecast(nn.Module): 
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x: (batch_size, seq_len, input_size)
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        out = self.fc(last_hidden)
        return out


A batch contains 64 windows.
Each window represents 7 days (168 hours), and each hour contains 9 features.

The LSTM is one shared model, not 64 different models. It processes the 64 windows in parallel.
For one window, the LSTM reads the 168 hours in temporal order. At each step it sees a vector of 9 features and updates an internal hidden state of size 64.

After hour 168, the final hidden state summarizes the previous week. That vector is passed through the final linear layer to forecast demand for the next hour.

The same operation happens for all 64 windows in the batch, producing 64 forecasts. Those forecasts are compared with the 64 true target values, the loss is computed, and the LSTM weights are updated.


In [ ]:
input_size = X_train_seq.shape[2]

model = LSTMForecast(input_size).to(device)

model

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)


In [ ]:
n_epochs = 15

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)

    train_loss /= len(train_loader.dataset)
        
    model.eval()
    validation_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in validation_loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            validation_loss += loss.item() * X_batch.size(0)
    validation_loss /= len(validation_loader.dataset)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.6f}, validation_loss={validation_loss:.6f}")

all_pred = []
test_loss = 0.0
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        test_loss += loss.item() * X_batch.size(0)
        all_pred.append(y_pred)

test_loss /= len(test_loader.dataset)
all_pred = torch.cat(all_pred, dim=0).cpu().numpy()
print(f"Final test_loss={test_loss:.6f}")


In [ ]:
y_test_true = scaler_y.inverse_transform(y_test_seq)
y_test_pred  = scaler_y.inverse_transform(all_pred)
y_test_pred.shape, y_test_true.shape

### Metric Calculation


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

mse = mean_squared_error(y_test_pred, y_test_true)
mae = mean_absolute_error(y_test_pred, y_test_true)
rmse = mse ** 0.5 

print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")  # same unit as the target


#### Plot


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,4))
plt.plot(y_test_pred[:500], label='Predicted')
plt.plot(y_test_true[:500], label='True')
plt.legend()
plt.show()

### Recursive Multi-Step Forecast


In [ ]:
model.eval()

sequence_length = X_test_seq.shape[1]
n_features = X_test_seq.shape[2]

def recursive_forecast_from_test(start_seq_idx, horizon=24):
    """
    Run a recursive multi-step forecast from X_test_seq[start_seq_idx].
    Returns predictions in the scaled [0, 1] range.
    This uses X_test_scaled for future covariates as a best-case assumption.
    """
    window = X_test_seq[start_seq_idx].copy()
    preds_val = []
    with torch.no_grad():
        for step in range(horizon):
            x_tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)
            y_pred = model(x_tensor)
            y_pred_val = y_pred.item()
            preds_val.append(y_pred_val)
            
            next_window = np.zeros_like(window)
            next_window[:-1, :] = window[1:, :]

            t_index = start_seq_idx + sequence_length + step
            future_feat = X_test_scaled[t_index].copy()
            future_feat[0] = y_pred_val
            next_window[-1, :] = future_feat

            window = next_window
    preds_val = np.array(preds_val)
    return preds_val


In [ ]:
start_seq_idx = 0
horizon = 48
preds_val_scaled = recursive_forecast_from_test(start_seq_idx, horizon)

preds_val = scaler_y.inverse_transform(preds_val_scaled.reshape(-1,1))

In [ ]:
# Extract actual values for the same period
true_start_index = start_seq_idx + sequence_length
y_true_scaled = y_test_scaled[true_start_index : true_start_index + horizon]
y_true = scaler_y.inverse_transform(y_true_scaled)

plt.figure(figsize=(12, 6))
plt.plot(y_true,label='Ground Truth (Actual Demand)', marker='.', color='black')
plt.plot(preds_val,label='Recursive Forecast (Model Prediction)', color='red', linestyle='--')
plt.title(f'Multi-step Forecast: Next {horizon} Hours')
plt.ylabel('Energy Demand (MW)')
plt.xlabel('Hours into the Future')
plt.legend()
plt.style.use('dark_background')
plt.grid(True, alpha=0.5)
plt.show()
